# 🚁 ArduPilot ↔ Isaac Sim bridge — launch in 2 clicks

All the code lives in [isaac_bridge.py](../isaac_bridge.py). Working configuration from 2026-07-08 (first successful NAV_TAKEOFF 5m): lockstep + Fix-12 (FLU→FRD frames) + Iris 1.5kg. Flight missions are in the neighboring `flight_missions.ipynb` notebook.

**Once per system** (PowerShell as admin) — two firewall rules:
```powershell
New-NetFirewallRule -DisplayName "ArduPilot JSON UDP 9002" -Direction Inbound -Protocol UDP -LocalPort 9002 -Action Allow
New-NetFirewallRule -DisplayName "ArduPilot MAVLink UDP 14550" -Direction Inbound -Protocol UDP -LocalPort 14550 -Action Allow
```

**Normal launch**: Isaac Sim open with a scene → "Import" cell → "BRIDGE" → "SITL" → diagnostics (packet count grows) → write missions in `flight_missions.ipynb`.

**After a crash**: Stop→Play in Isaac → `await reset_motors()` → `stop_sitl()` → `launch_sitl()` → run the mission again.

## 1. Import

In [ ]:
import sys, os
# Find the repo root (folder containing isaac_bridge.py) from any cwd, so the
# project runs from ANY clone location without editing this path.
_root = os.getcwd()
for _ in range(4):
    if os.path.exists(os.path.join(_root, "isaac_bridge.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
import importlib
import isaac_bridge
importlib.reload(isaac_bridge)   # pick up module edits without restarting the kernel
from isaac_bridge import *
print("isaac_bridge loaded from", _root)

## 2. BRIDGE — bring everything up with one command (9 steps, safe to re-run)

In [ ]:
await bridge_up()

## 3. SITL — launch ArduPilot (a separate window will open)

In [ ]:
launch_sitl()

## Diagnostics (run as often as you like: packets should be growing, dt≈4.17ms)

In [ ]:
await diag()

## Utilities: reset after Stop→Play / restart SITL

In [ ]:
await reset_motors()

In [ ]:
stop_sitl()

## Sign test (frame diagnostics, WITHOUT SITL!)

A historical tool that found the root of the "spinning top" (Isaac world: Y=West). Use it whenever the signs are in doubt after changing the scene/model. SITL must be OFF (`stop_sitl()`).

In [ ]:
import asyncio

async def _srv(m1, m2, m3, m4):
    await execute_in_isaac("_wsl_servos[:] = [%d,%d,%d,%d] + [1000]*12" % (m1, m2, m3, m4))

async def _tl(cmd):
    await execute_in_isaac("import omni.timeline as _t; _t.get_timeline_interface().%s()" % cmd)

r1 = await execute_in_isaac("print(_ap_pkt_count)")
await asyncio.sleep(1.0)
r2 = await execute_in_isaac("print(_ap_pkt_count)")
if r1.get("output") != r2.get("output"):
    print("STOP: SITL is running (packets flowing). Run stop_sitl() and start the test again.")
else:
    tests = [
        ("YAW+   (top M1,M2 - 'yaw right, clockwise from above')", (1700, 1700, 1300, 1300)),
        ("ROLL+  (top M2,M3 - 'right side down')",                 (1300, 1700, 1700, 1300)),
        ("PITCH+ (top M1,M3 - 'nose up')",                         (1700, 1300, 1700, 1300)),
    ]
    print("Sign test: 3 probes, each with a scene reset...")
    for name, pat in tests:
        await _srv(1000, 1000, 1000, 1000)
        await _tl("stop")
        await asyncio.sleep(0.7)
        await _tl("play")
        await asyncio.sleep(0.7)
        await _srv(1500, 1500, 1500, 1500)
        await asyncio.sleep(0.35)
        await _srv(*pat)
        await asyncio.sleep(0.3)
        r = await execute_in_isaac(
            "print(_wsl_log[-1]['gyro_deg'], '| tilt', _wsl_log[-1]['tilt_deg'], 'deg | alt', _wsl_log[-1]['alt'])"
        )
        await _srv(1000, 1000, 1000, 1000)
        print()
        print(name)
        print("   body gyro [x,y,z] deg/s (z-up frame):", r.get("output", "").strip())
    await _tl("stop")
    await asyncio.sleep(0.5)
    await _tl("play")
    await _srv(1000, 1000, 1000, 1000)
    print()
    print("Done.")